# **From Reactive to Predictive: ML Applications in Civil Infrastructure Maintenance**

## **DAT3000B Group 8 Project**

**Session:** Winter 2025/2026

**Authors:** Memeh Vanessa, Niu Zi Yuan, Piratheeparatnam Arjayn, Pristav Tudor-Mihai  

<br>
<br>


## **Project Overview**
The 2025 National Bridge Inventory (NBI) report indicates a significant portion of U.S. infrastructure is approaching the end of its design life. Traditional maintenance is often reactive—repairing bridges only after significant deterioration is detected.

### **Our Engineering Goal**

To transition from reactive to predictive maintenance. By analyzing the 2025 dataset (~600k entries), we aim to predict a bridge's "Condition Rating" based on its age, geography (climate zone), and design type. This model is designed to allow engineers to forecast which regions require immediate budget allocation before public safety is compromised.

### **Dataset Source**
This project utilizes the **2025 National Bridge Inventory (NBI)** dataset, sourced directly from the Federal Highway Administration's **LTBP InfoBridge** data portal. The complete national dataset contains approximately 600,000 entries. To avoid hardware constraints and prevent the model from overfitting to specific state budgets, we are working with a geographically diverse subset of 20 states, representing five core climate zones.

# 1. Project Setup and Imports

n this first step, we are importing all the necessary libraries for data manipulation, machine learning, and evaluation. We also set a global random seed to ensure that our data splits and model training results are reproducible across the team.

In [10]:
# Core
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Splitting / Validation
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Models
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyClassifier, DummyRegressor

# Evaluation & Visualizations
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.decomposition import PCA
from sklearn.inspection import DecisionBoundaryDisplay

# Global random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


ImportError: cannot import name 'DecisionBoundaryDisplay' from 'sklearn.inspection' (/opt/anaconda3/lib/python3.9/site-packages/sklearn/inspection/__init__.py)

# 2. Data Loading & Initial Cleaning

Government datasets often have formatting quirks. Here, we load the raw CSV data and clean the column names so they are easier to call in Python. We also strip out any trailing white spaces or extra quotation marks from the string columns (like the state names and materials) so our categorical encoders don't treat 'Arizona' and 'Arizona ' as two different things. Finally, we drop any rows where our target variable (CAT10 - Bridge Condition) is missing, as we cannot train a model on null targets.

In [2]:

# --- File Merging ---
# Have the files saved by STATE_NAME.txt EG: Arizona.txt

# List of your 20 specific file names
file_names = [
    'Arizona.txt',
    'Utah.txt', 'SouthCarolina.txt', 'Pennsylvania.txt', 'Ohio.txt',
    'NewYork.txt', 'NewMexico.txt', 'Nevada.txt', 'Michigan.txt',
    'Louisiana.txt', 'Hawaii.txt', 'Florida.txt',

    'Washington.txt', 'Oregon.txt', 'California.txt', 'Alaska.txt', 'NorthDakota.txt',
    'Minnesota.txt', 'Wyoming.txt',
] # mpdate the 'State_X' with actual file names when the others roll in

dataframes = []

print("Loading and merging files...")
for file in file_names:
    try:
        # quotechar="'" to handle single-quote text wrapping
        # on_bad_lines='skip' to drop irreparably broken rows instead of crashing
        temp_df = pd.read_csv(file, quotechar="'", on_bad_lines='skip')
        dataframes.append(temp_df)
        print(f"Loaded {file} - Shape: {temp_df.shape}")
    except FileNotFoundError:
        print(f"Warning: {file} not found. Double-check spelling!")

# merge them into a single dataframe named 'df'
df = pd.concat(dataframes, ignore_index=True)
print(f"\nMaster Dataset Shape after merge: {df.shape}")


# --- Data Cleaning ---
print("\nCleaning data...")

# clean column names (make lowercase, replace ' - ' and spaces with underscores, remove periods)
df.columns = df.columns.str.lower().str.replace(' - ', '_').str.replace(' ', '_').str.replace('.', '')

# clean string columns (strip trailing whitespaces and remove single quotes just in case)
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip().str.replace("'", "")

# drop rows where the target variable 'cat10_bridge_condition' is NaN
df = df.dropna(subset=['cat10_bridge_condition'])

print(f"Final Cleaned Dataset Shape: {df.shape}\n")

# print df.info() and df.head() to confirm everything merged correctly
df.info()
display(df.head())

Loading and merging files...


ValueError: No objects to concatenate

# 3. Geographic Feature Construction

To prevent the model from simply memorizing the maintenance budgets of individual states, we engineer a new feature: climate_zone. We map the 20 chosen states into five distinct environmental categories based on how weather physically degrades infrastructure: Hot & Arid (h_a), Humid & Coastal (h_c), Freeze-Thaw (f_t), Constant Moisture (c_m), and Deep Freeze (d_f).

In [ ]:


print("Constructing Geographic Features...")

# Dictionary mapping the raw state names to our 5 engineered climate zones
climate_map = {
    # Hot & Arid
    'Arizona': 'h_a', 'Nevada': 'h_a', 'New Mexico': 'h_a', 'Utah': 'h_a',

    # Humid & Coastal
    'Florida': 'h_c', 'Louisiana': 'h_c', 'South Carolina': 'h_c', 'Hawaii': 'h_c',

    # Freeze-Thaw & Road Salt
    'Ohio': 'f_t', 'Michigan': 'f_t', 'Pennsylvania': 'f_t', 'New York': 'f_t',

    # Constant Moisture
    'Washington': 'c_m', 'Oregon': 'c_m', 'California': 'c_m',

    # Deep Freeze
    'Alaska': 'd_f', 'North Dakota': 'd_f', 'Minnesota': 'd_f', 'Wyoming': 'd_f'
}

# create the new 'climate_zone' column using the mapping dictionary
df['climate_zone'] = df['1_state_name'].map(climate_map)

# drop unmapped states
initial_rows = len(df)
df = df.dropna(subset=['climate_zone'])
dropped_rows = initial_rows - len(df)
if dropped_rows > 0:
    print(f"Dropped {dropped_rows} rows due to unmapped states.")

# print the value counts of the new 'climate_zone' column clearly
print("\n--- Distribution of Bridges by Climate Zone ---")
counts = df['climate_zone'].value_counts()
percentages = df['climate_zone'].value_counts(normalize=True) * 100

# format the output for easy reading
for zone in counts.index:
    print(f"{zone.upper()}: {counts[zone]:,} bridges ({percentages[zone]:.2f}%)")

print("-" * 34)

# 4. Exploratory Data Analysis (EDA) & Target Distribution

Before splitting or modeling, we need to visually explore our dataset to understand the relationships at play. We will plot the distribution of our target variable to identify class imbalances. We will also generate scatter plots and bar charts to see how continuous variables (like Bridge Age or Daily Traffic) correlate with the Bridge Condition across different climate zones.

In [ ]:

print("Calculating Target Distributions...")

# define target column name
target_col = 'cat10_bridge_condition'

# compute counts and % for the target variable
condition_counts = df[target_col].value_counts()
condition_percentages = df[target_col].value_counts(normalize=True) * 100

# print result
print("\n--- Target Variable Distribution ---")
for condition in condition_counts.index:
    print(f"{condition}: {condition_counts[condition]:,} bridges ({condition_percentages[condition]:.2f}%)")
print("-" * 34)

# --- General Insights ---
print("\n------- General Insights -------")
# 1. age extremes
median_age_by_climate = df.groupby('climate_zone')['bridge_age_(yr)'].median()
oldest_climate = median_age_by_climate.idxmax()
youngest_climate = median_age_by_climate.idxmin()
print(f"-> OLDEST INFRASTRUCTURE: The {oldest_climate.upper()} climate zone has the oldest median bridge age ({median_age_by_climate.max():.1f} years).")
print(f"-> YOUNGEST INFRASTRUCTURE: The {youngest_climate.upper()} climate zone has the youngest median bridge age ({median_age_by_climate.min():.1f} years).")

# filter to only 'Poor' bridges
poor_bridges = df[df['cat10_bridge_condition'] == 'Poor']

# calculate mean age of 'Poor' bridges per climate zone
mean_age_poor = poor_bridges.groupby('climate_zone')['bridge_age_(yr)'].mean().reset_index()
mean_age_poor = mean_age_poor.sort_values(by='bridge_age_(yr)')

fastest_decay = mean_age_poor.iloc[0]
slowest_decay = mean_age_poor.iloc[-1]
lifespan_gap = slowest_decay['bridge_age_(yr)'] - fastest_decay['bridge_age_(yr)']


# 2. traffic volume
traffic_by_climate = df.groupby('climate_zone')['29_average_daily_traffic'].mean()
highest_traffic_climate = traffic_by_climate.idxmax()
print(f"-> HEAVIEST TRAFFIC: The {highest_traffic_climate.upper()} zone handles the highest average daily traffic ({traffic_by_climate.max():,.0f} vehicles/day).")
print("-" * 34)

print("\n------------------------------------------")
fastest_decay = mean_age_poor.iloc[0]
slowest_decay = mean_age_poor.iloc[-1]
lifespan_gap = slowest_decay['bridge_age_(yr)'] - fastest_decay['bridge_age_(yr)']

print(f"-> MOST VULNERABLE: Bridges in the {fastest_decay['climate_zone'].upper()} zone reach 'Poor' condition fastest, at an average age of {fastest_decay['bridge_age_(yr)']:.1f} years.")
print(f"-> MOST DURABLE: Bridges in the {slowest_decay['climate_zone'].upper()} zone last the longest, reaching 'Poor' condition at an average age of {slowest_decay['bridge_age_(yr)']:.1f} years.")
print(f"-> THE CLIMATE GAP: The environment causes a {lifespan_gap:.1f}-year difference in average structural lifespan.")
print("-" * 65)


print("\nGenerating EDA Visualizations...")

# map specific colors to the conditions
custom_colors = {'Good': 'forestgreen', 'Fair': 'royalblue', 'Poor': 'red'}

# force the boxplot outliers (fliers) to be highly visible black circles
black_circle_outliers = dict(marker='o', markerfacecolor='black', markeredgecolor='black', markersize=4, alpha=0.6)

# create a figure with 3 subplots for the visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PLOT 1: Seaborn countplot showing the distribution of Bridge Conditions
sns.countplot(
    data=df,
    x=target_col,
    order=['Good', 'Fair', 'Poor'],
    palette=custom_colors,
    ax=axes[0]
)
axes[0].set_title('Distribution of Bridge Conditions')
axes[0].set_xlabel('Condition')
axes[0].set_ylabel('Count')

# PLOT 2: Scatter plot of Bridge Age vs Average Daily Traffic
# reduced sample size to prevent from crashing
sample_df = df.sample(n=min(25000, len(df)), random_state=RANDOM_STATE)
sns.scatterplot(
    data=sample_df,
    x='bridge_age_(yr)',
    y='29_average_daily_traffic',
    hue=target_col,
    hue_order=['Good', 'Fair', 'Poor'],
    palette=custom_colors,
    alpha=0.6,
    ax=axes[1]
)
axes[1].set_title('Bridge Age vs. Daily Traffic (Sampled)')
axes[1].set_xlabel('Bridge Age (Years)')
axes[1].set_ylabel('Average Daily Traffic')

## i think there is a typo in one of the datasets under the age so cap at 250
axes[1].set_xlim(0, 250)

# PLOT 3: Boxplot showing Bridge Age distribution across the climate zones
sns.boxplot(
    data=df,
    x='climate_zone',
    y='bridge_age_(yr)',
    palette='Set2',
    flierprops=black_circle_outliers,
    ax=axes[2]
)
axes[2].set_title('Bridge Age by Climate Zone')
axes[2].set_xlabel('Climate Zone')
axes[2].set_ylabel('Bridge Age (Years)')

# adjust layout so the plots don't overlap and display them
plt.tight_layout()
plt.show()

# =====================================================================

# --- Section 4.1: Hypothesis Testing ---

print("\nGenerating Hypothesis Testing Visualizations...")

# Filter to only (Poor) bridges to find the average age they fail
poor_bridges = df[df['cat10_bridge_condition'] == 'Poor']
mean_age_poor = poor_bridges.groupby('climate_zone')['bridge_age_(yr)'].mean().reset_index()
mean_age_poor = mean_age_poor.sort_values(by='bridge_age_(yr)')

# --- Hypothesis Interpretations ---
print("\n--- Hypothesis Insights ---")
fastest_decay_zone = mean_age_poor.iloc[0]['climate_zone']
fastest_decay_age = mean_age_poor.iloc[0]['bridge_age_(yr)']

slowest_decay_zone = mean_age_poor.iloc[-1]['climate_zone']
slowest_decay_age = mean_age_poor.iloc[-1]['bridge_age_(yr)']

print(f"-> FASTEST DEGRADATION: Bridges in the {fastest_decay_zone.upper()} zone reach a 'Poor' condition fastest, surviving on average only {fastest_decay_age:.1f} years.")
print(f"-> LONGEST LIFESPAN: Bridges in the {slowest_decay_zone.upper()} zone survive the longest, reaching 'Poor' condition at an average age of {slowest_decay_age:.1f} years.")
print("-" * 37)


# create a figure with 2 subplots
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# --- PLOT 1: Age by Condition by Climate Zone (Grouped Boxplot) ---
# (This was your old Plot 2 with the custom colors)
sns.boxplot(
    data=df,
    x='climate_zone',
    y='bridge_age_(yr)',
    hue='cat10_bridge_condition',
    hue_order=['Good', 'Fair', 'Poor'],
    palette=custom_colors,
    flierprops=black_circle_outliers,
    ax=axes[0]
)
axes[0].set_title('Bridge Age by Condition and Climate Zone', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Climate Zone', fontsize=12)
axes[0].set_ylabel('Bridge Age (Years)', fontsize=12)
axes[0].legend(title='Condition', loc='upper right')

# --- PLOT 2: Average Decay Point (Bar Chart) ---
# (This was your old Plot 3)
sns.barplot(
    data=mean_age_poor,
    x='climate_zone',
    y='bridge_age_(yr)',
    palette='Reds_r',
    ax=axes[1]
)
axes[1].set_title('Average Age of "Poor" Bridges Across Climate Zones', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Climate Zone', fontsize=12)
axes[1].set_ylabel('Average Age at "Poor" Condition (Years)', fontsize=12)

# add the exact age numbers floating on top of the bars for your final report
for p in axes[1].patches:
    axes[1].annotate(format(p.get_height(), '.1f') + ' yrs',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha = 'center', va = 'center',
                     xytext = (0, 9),
                     textcoords = 'offset points',
                     fontweight='bold')

plt.tight_layout()
plt.show()

# --- Section 4.2: Engineering Data Pulling & Feature Correlation ---

print("Querying Material and Agency Decay Rates...\n")

# 1. MATERIAL INSIGHTS
# calculate the percentage of 'Poor' bridges per material
# filter out materials that have very few bridges (e.g., less than 500) to avoid skewed 100% failure rates on rare materials
material_counts = df['43a_main_span_material'].value_counts()
valid_materials = material_counts[material_counts > 500].index
df_mat = df[df['43a_main_span_material'].isin(valid_materials)]

mat_ct = pd.crosstab(df_mat['43a_main_span_material'], df_mat['cat10_bridge_condition'], normalize='index') * 100
mat_poor = mat_ct['Poor'].sort_values(ascending=False)

print("--- TOP 3 MATERIALS WITH HIGHEST FAILURE ('POOR') RATES ---")
for i, (material, pct) in enumerate(mat_poor.head(3).items(), 1):
    print(f"{i}. {material}: {pct:.2f}% rated Poor")

print("\n--- TOP 3 MATERIALS WITH LOWEST FAILURE ('POOR') RATES ---")
# sort ascending to get the lowest failure rates
mat_poor_lowest = mat_ct['Poor'].sort_values(ascending=True)
for i, (material, pct) in enumerate(mat_poor_lowest.head(3).items(), 1):
    print(f"{i}. {material}: {pct:.2f}% rated Poor")


# 2. INSIGHTS
# calculate the % of 'Poor' bridges per owner agency
agency_counts = df['22_owner_agency'].value_counts()
valid_agencies = agency_counts[agency_counts > 500].index
df_own = df[df['22_owner_agency'].isin(valid_agencies)]

own_ct = pd.crosstab(df_own['22_owner_agency'], df_own['cat10_bridge_condition'], normalize='index') * 100
own_poor = own_ct['Poor'].sort_values(ascending=False)

print("\n--- TOP 3 AGENCIES WITH HIGHEST MAINTENANCE DISPARITY (FAILURE RATES) ---")
for i, (agency, pct) in enumerate(own_poor.head(3).items(), 1):
    print(f"{i}. {agency}: {pct:.2f}% rated Poor")
print("-" * 50)


# 3. FEATURE CORRELATION MATRIX
print("\nCalculating Feature Correlations...")

# must temporarily map Condition to a number to correlate iwth numbers liek Age and Traffic
# Poor = 0, Fair = 1, Good = 2 (So a negative correlation means as the feature goes UP, the condition goes DOWN to 'Poor')
df['condition_numeric'] = df['cat10_bridge_condition'].map({'Poor': 0, 'Fair': 1, 'Good': 2})

# select only the core numeric columns
numeric_cols = [
    'bridge_age_(yr)',
    '29_average_daily_traffic',
    '45_number_of_spans_in_main_unit',
    '49_structure_length_(ft)',
    'cat29_deck_area_(sq_ft)',
    'condition_numeric'
]

# calculate the correlation matrix
corr_matrix = df[numeric_cols].corr()

# isolate the correlations (specifically related to the Bridge Condition)
condition_correlations = corr_matrix['condition_numeric'].drop('condition_numeric').sort_values()

print("\n--- FEATURES MOST CORRELATED WITH BRIDGE DECAY ---")
print("(Note: Negative values indicate that as the feature increases, the bridge condition worsens toward 'Poor')")
for feature, corr_value in condition_correlations.items():
    print(f"- {feature}: {corr_value:.3f}")

# plot the correlation matrix as a Heatmap so you can see how features relate to each other
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title("Feature Correlation Matrix", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# drop the temporary numeric condition column -  prevent data leakage
df = df.drop(columns=['condition_numeric'])

# 5. Stratified Train | Validation | Test Splitting

To properly evaluate our models, we implement a rigorous three-way split: 60% Training, 20% Validation (for hyperparameter tuning), and 20% Testing (isolated for final evaluation). Crucially, we stratify these splits based on our engineered climate_zone feature. This ensures that every geographic region is proportionally represented in all three datasets, preventing geographic bias during training.

In [ ]:

from sklearn.model_selection import train_test_split

print("Preparing Data for Machine Learning...")

# CRITICAL FIX: keep ONLY rows where the condition is strictly (Good) (Fair) (Poor)
valid_conditions = ['Good', 'Fair', 'Poor']
df = df[df['cat10_bridge_condition'].isin(valid_conditions)]

# downsample the ENTIRE dataset to a memory-safe size (20% = ~1.18 million rows)
# BEFORE splitting, perfectly stratified by climate zone.
print("Downsampling master dataset to 20% to guarantee Colab RAM safety...")
df_safe, _ = train_test_split(
    df, train_size=0.20, random_state=RANDOM_STATE, stratify=df['climate_zone']
)

# define X (features) and y (target) from the SAFE dataset
# drop bridge IDs, city names, and coordinates to prevent data leakage/memorization
features_to_keep = [
    'bridge_age_(yr)',
    '29_average_daily_traffic',
    '45_number_of_spans_in_main_unit',
    '49_structure_length_(ft)',
    'cat29_deck_area_(sq_ft)',
    'climate_zone',
    '43a_main_span_material',
    '43b_main_span_design',
    '22_owner_agency'
]

X = df_safe[features_to_keep]
y = df_safe['cat10_bridge_condition']

print(f"Total valid samples for ML: {len(X):,}")

# create a 60/40 train/temp split, stratified by X['climate_zone']
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=X['climate_zone']
)

# 50/50 split of the temp data into validation and test sets
# results in a final 60/20/20 overall split of our safe subset
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=X_temp['climate_zone']
)

# print the shapes of X_train, X_val, and X_test
print("\n--- Split Shapes ---")
print(f"Training Set:   {X_train.shape[0]:,} rows")
print(f"Validation Set: {X_val.shape[0]:,} rows")
print(f"Test Set:       {X_test.shape[0]:,} rows")

# 6. Preprocessing Pipeline

Bridge data contains a mix of numerical physical metrics (like Age, Traffic, and Deck Area) and categorical descriptions (like Design and Material). We build a preprocessing pipeline using ColumnTransformer to handle both simultaneously. Missing numerical values are imputed with the median and scaled using StandardScaler, while missing categorical values are filled with the mode and encoded using OneHotEncoder.

In [ ]:

print("Building Preprocessing Pipeline...")

# numeric feature column names
numeric_features = [
    'bridge_age_(yr)',
    '29_average_daily_traffic',
    '45_number_of_spans_in_main_unit',
    '49_structure_length_(ft)',
    'cat29_deck_area_(sq_ft)'
]

# categorical feature column names (include 'climate_zone')
categorical_features = [
    'climate_zone',
    '43a_main_span_material',
    '43b_main_span_design',
    '22_owner_agency'
]

# build numeric_transformer pipeline (Impute median -> StandardScaler)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# build categorical_transformer pipeline (Impute mode -> OneHotEncoder)
# handle_unknown='ignore' prevents errors if the validation|test set has a weird string not seen in training or just funky behaviour
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# assemble using ColumnTransformer and store in variable `preprocessor`
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Pipeline built successfully! Ready for baseline modeling.")

# 7. Baseline Modelling

To measure if our complex machine learning algorithms are actually finding underlying patterns, we must first establish baseline performance metrics. We will create a naive classification baseline that simply predicts the majority class every time, and a naive regression baseline that predicts the mean age/condition.

In [ ]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
import time

print("Transforming data through the preprocessing pipeline...")
start_time = time.time()

# pass the data through the ColumnTransformer (Fit on Train, Transform on Val)
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

print(f"Data transformed in {time.time() - start_time:.1f} seconds.\n")

print("--- BASELINE MODEL RESULTS (Dummy Classifier) ---")
# initialize and fit a DummyClassifier
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train_processed, y_train)

# predict on validation set
y_pred_dummy = dummy_clf.predict(X_val_processed)

# print metrics
print(f"Overall Accuracy: {accuracy_score(y_val, y_pred_dummy) * 100:.2f}%")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_dummy) * 100:.2f}%\n")

# zero_division=0 prevents warnings since the model never guesses 'Fair' or 'Poor'
print(classification_report(y_val, y_pred_dummy, zero_division=0))

# 8. Classification Model (XGBoost with L1 Regularization)

For our primary classification task, we use Extreme Gradient Boosting (XGBoost). Because one-hot encoding expands our categorical features significantly, we use LASSO (L1 Regularization) via the reg_alpha parameter in XGBoost. This acts as an embedded feature selector, automatically driving the weights of irrelevant physical or geographic metrics to zero, preventing overfitting and highlighting the most critical predictors of bridge failure.

In [ ]:

### I STILL NEED TO TWEAK THE MODEL MORE -- COULD BE IMPROVED FOR BETTER OVERALL ACCURACY
## PLS DO NOT TOUCH

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
import time
from sklearn.utils.class_weight import compute_sample_weight

print("Mapping target variables for XGBoost...")
# map string targets (Good, Fair, Poor) to integers (2, 1, 0) for XGBoost
target_mapping = {'Poor': 0, 'Fair': 1, 'Good': 2}
y_train_encoded = y_train.map(target_mapping)
y_val_encoded = y_val.map(target_mapping)

print("Calculating balanced sample weights for the minority class ('Poor')...")

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_encoded)

print("Building XGBoost Pipeline")
# build a Pipeline containing the `preprocessor` and `XGBClassifier`
# use the reg_alpha parameter in XGBoost for L1 Regularization
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=9,
        random_state=RANDOM_STATE,
        # keeps Colab RAM from crashing
        tree_method='hist',
        # L1 regularization
        # reg_alpha=1.0,
        n_jobs=-1
    ))
])

print("Training XGBoost Pipeline (This will take a minute or two)...")
xgb_start = time.time()

# fit the pipeline on the training data -  passing sample weights directly to the XGBoost classifie
xgb_pipeline.fit(X_train, y_train_encoded, classifier__sample_weight=sample_weights)

print(f"XGBoost Pipeline training complete in {time.time() - xgb_start:.1f} seconds.\n")

print("Generating predictions on Validation Set...")
# predict using the unified pipeline
y_pred_xgb_encoded = xgb_pipeline.predict(X_val)

# reverse the mapping (0->Poor, 1->Fair, 2->Good) to print readable metrics
reverse_mapping = {0: 'Poor', 1: 'Fair', 2: 'Good'}
y_pred_xgb = [reverse_mapping[val] for val in y_pred_xgb_encoded]

print("--- XGBOOST MODEL RESULTS ---")
print(f"Overall Accuracy: {accuracy_score(y_val, y_pred_xgb) * 100:.2f}%")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_xgb) * 100:.2f}%\n")
print(classification_report(y_val, y_pred_xgb))

# 9. Classification Evaluation & Visualization

We evaluate the XGBoost model on the Validation set using imbalance-aware metrics. To make the results interpretable, we generate a Confusion Matrix to see exactly where the model misclassifies data. We will also visualize the decision boundary using Principal Component Analysis (PCA) to observe how the algorithm geometrically separates the condition classes.

In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.inspection import DecisionBoundaryDisplay
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

# --- Standard Metrics ---
# TODO: Generate predictions on the validation set
y_pred_xgb_encoded = xgb_pipeline.predict(X_val)

# TODO: Compute and print F1-score and Balanced Accuracy (use encoded for numerical data)
f1 = f1_score(y_val_encoded, y_pred_xgb_encoded, average='weighted')
balanced_accuracy = balanced_accuracy_score(y_val_encoded, y_pred_xgb_encoded)
print("Balanced accuracy:", balanced_accuracy)
print("F1 score:", f1)

# TODO: Extract and print the top Feature Importances from the XGBoost model
feature_names = xgb_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = xgb_pipeline.named_steps['classifier'].feature_importances_
# make dataframe
frame = pd.DataFrame({"Feature names": feature_names, "Importance rating": importances})
sorted_df = frame.sort_values(by="Importance rating", ascending=False)
# print most relevant features from model
print("\nTop 5 most relevant features in XGBoost:")
print(sorted_df.head(5))

# --- Visual 1: Confusion Matrix ---
# TODO: Compute the confusion matrix with clear labels (Good, Fair, Poor)
plotConfusionMatrix(y_val_encoded, y_pred_xgb_encoded)

# --- Visual 2: PCA Decision Boundary ---
# To plot a 2D boundary, we must reduce the processed features to 2 principal components.
# TODO: Transform X_train using the preprocessor
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)

# TODO: Fit PCA(n_components=2) on the processed training data.
pca = PCA(n_components=2, random_state=RANDOM_STATE)
#(Fit+transform the train set, but only transform validation set)
X_train_pca = pca.fit_transform(X_train_processed) #note X_train_pca is encoded not string
X_val_pca = pca.transform(X_val_processed)

# TODO: Calculate and print the Explained Variance
explained_var = pca.explained_variance_ratio_
print("Explained variance:", explained_var)

# TODO: Train a secondary, simplified XGBClassifier on just those 2 PCA components
xgb_simplified = XGBClassifier(
    n_estimators = 200,
    learning_rate = 0.1,
    max_depth = 5,
    random_state = RANDOM_STATE
)
xgb_simplified.fit(X_train_pca, y_train_encoded)

# TODO: Use DecisionBoundaryDisplay.from_estimator to plot the 2D boundary
plt.figure()
DecisionBoundaryDisplay.from_estimator(
    xgb_simplified,
    X_train_pca,
)
# TODO: Scatter plot the actual PCA-transformed validation points on top of the boundary
scatter = plt.scatter(
    X_val_pca[:, 0],
    X_val_pca[:, 1],
    c=y_val_encoded
)

# This is a legend bar to show what the colours in the points actually mean
cbar = plt.colorbar(scatter)
cbar.set_ticks([0, 1, 2])
cbar.set_ticklabels(['Poor', 'Fair', 'Good'])

# Label and show plot
plt.xlabel("pc1")
plt.ylabel("pc2")
plt.title("2D Decision Boundary")
plt.show()

# 10. Regression Analysis (Continuous Decay Modelling)

To statistically verify if bridges in coastal or harsh climates degrade at a faster rate, we build a Multiple Linear Regression model. We map the condition ratings to continuous values (Good=2, Fair=1, Poor=0) to model the continuous rate of decay against bridge age and climate factors. Evaluating Mean Squared Error (MSE) vs. Mean Absolute Error (MAE) will help us identify severe prediction outliers—bridges that the model thinks are safe but are actually in critical condition.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Mapping to integers is needed (0->Poor, 1->Fair, 2->Good)
mlr_mapping = {'Poor': 0, 'Fair': 1, 'Good': 2}
y_train_mlr = y_train.map(mlr_mapping)
y_val_mlr = y_val.map(mlr_mapping)


# TODO: Build a Pipeline containing the `preprocessor` (from above) and `LinearRegression`
mlr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
# TODO: Fit the pipeline on the training data
mlr_pipeline.fit(X_train, y_train_mlr)

# Find R^2, MSE and MAE
y_mlr_pred = mlr_pipeline.predict(X_val)
mlr_r_squared = r2_score(y_val_mlr, y_mlr_pred)
mlr_mse = mean_squared_error(y_val_mlr, y_mlr_pred)
mlr_mae = mean_absolute_error(y_val_mlr, y_mlr_pred)

#Display
print("R^2: ", mlr_r_squared)
print("MSE: ", mlr_mse)
print("MAE: ", mlr_mae)

# 11. Final Test Evaluation & Outlier Analysis

We use the isolated Test set one time to perform a final, unbiased evaluation of our XGBoost and Linear Regression models. By comparing the regression metrics (MAE vs. MSE), we can determine if the model is making a few massively wrong predictions (which exponentially inflates MSE) or consistent small errors. Predicting a "Poor" bridge to be "Good" is a massive safety hazard, making outlier analysis critical.

In [ ]:
# --- XGBoost Final Evaluation ---

# TODO: Predict on X_test using the XGBoost pipeline
y_pred_test_xgb_encoded = xgb_pipeline.predict(X_test)

# Convert numeric predictions back to original labels
y_pred_test_xgb = [reverse_mapping[val] for val in y_pred_test_xgb_encoded]

# TODO: Print classification metrics for the test set
print("--- XGBoost Final Test Results ---")
print("Overall Accuracy:", accuracy_score(y_test, y_pred_test_xgb))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_test_xgb))
print(classification_report(y_test, y_pred_test_xgb))


# --- Linear Regression Final Evaluation ---

# TODO: Predict on X_test using the Linear Regression pipeline
y_pred_test_mlr = mlr_pipeline.predict(X_test)

# Convert y_test to numeric form for regression metrics
y_test_mlr = y_test.map(mlr_mapping)

# TODO: Compute and print Mean Squared Error (MSE)
test_mse = mean_squared_error(y_test_mlr, y_pred_test_mlr)
print("--- Linear Regression Final Test Results ---")
print("MSE:", test_mse)

# TODO: Compute and print Mean Absolute Error (MAE)
test_mae = mean_absolute_error(y_test_mlr, y_pred_test_mlr)
print("MAE:", test_mae)

# Final Analysis & Conclusion


##Model Performance
Which model performed best and why? Compare the XGBoost performance to the Baseline Classifier, and the Linear Regression to the Baseline Regressor.





##Imbalance Impact
How did class imbalance affect metric choice?

##Decay Rates
Did coastal/harsh climates show a statistically faster rate of decay based on the regression weights?
